In [ ]:
# CELL 1: Set up a clean, fixed Python environment for the benchmark.
!pip -q uninstall -y numpy scipy scikit-learn sklearn transformers pandas duckdb sqlglot bitsandbytes accelerate timm datasets sentencepiece protobuf safetensors psutil tqdm huggingface_hub

!pip -q install --no-cache-dir \
    "numpy==1.26.4" \
    "scipy==1.13.1" \
    "scikit-learn==1.5.2" \
    "pandas==2.2.3" \
    "sqlglot==26.31.0" \
    "transformers>=4.57.0" \
    "accelerate" \
    "bitsandbytes" \
    "timm" \
    "datasets" \
    "sentencepiece" \
    "protobuf" \
    "safetensors" \
    "psutil" \
    "tqdm" \
    "huggingface_hub"

print("Install finished. Restart the runtime before running the next cell.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 256.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 109.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 288.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 310.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 292.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 198.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 106.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 272.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 474.3/474.3 kB 307.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 106.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 441.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# CELL 2: Check that the key libraries loaded correctly after restarting the runtime.
import numpy
import transformers
import pandas
import sqlglot
import torch
import huggingface_hub

print("numpy:", numpy.__version__)
print("transformers:", transformers.__version__)
print("pandas:", pandas.__version__)
print("sqlglot:", sqlglot.__version__)
print("torch:", torch.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

numpy: 1.26.4
transformers: 5.8.0
pandas: 2.2.3
sqlglot: 26.31.0
torch: 2.10.0+cu128
huggingface_hub: 1.14.0
cuda available: True
gpu: Tesla T4


In [ ]:
# CELL 3: Load the main libraries and define the project paths, models, and benchmark settings.
import os
import re
import gc
import json
import math
import time
import shutil
import random
import warnings
import sqlite3
import signal
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple

import numpy as np
import pandas as pd
import psutil
import sqlglot
from sqlglot import exp
from tqdm.auto import tqdm

import torch
from huggingface_hub import login, hf_hub_download, whoami
from transformers import (
    AutoTokenizer,
    AutoProcessor,
    AutoModelForCausalLM,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path("/content/drive/MyDrive/nl2sql_sqlite")
ROOT.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = Path("/content/nl2sql_hf_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

SPIDER_ROOT = Path("/content/drive/MyDrive/data/spider")

print("ROOT:", ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print("CACHE_DIR:", CACHE_DIR)
print("SPIDER_ROOT:", SPIDER_ROOT)
print("Root exists:", SPIDER_ROOT.exists())
print("dev.json exists:", (SPIDER_ROOT / "dev.json").exists())
print("tables.json exists:", (SPIDER_ROOT / "tables.json").exists())
print("database dir exists:", (SPIDER_ROOT / "database").exists())

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

MAX_NEW_TOKENS = 256
BATCH_SIZE = 1
TOP_K_TABLES = 4
SAMPLE_VALUES_PER_TEXT_COL = 3
SQLITE_TIMEOUT_SEC = 3.0
SQLITE_PROGRESS_STEPS = 1000

MODEL_REGISTRY = [
    {"name": "Qwen3-0.6B",               "model_id": "Qwen/Qwen3-0.6B",                     "family": "qwen",    "loader": "causal",            "chat_kwargs": {"enable_thinking": False}},
    {"name": "Qwen3-1.7B",               "model_id": "Qwen/Qwen3-1.7B",                     "family": "qwen",    "loader": "causal",            "chat_kwargs": {"enable_thinking": False}},
    {"name": "Qwen3-4B-Instruct-2507",   "model_id": "Qwen/Qwen3-4B-Instruct-2507",         "family": "qwen",    "loader": "causal",            "chat_kwargs": {}},
    {"name": "Gemma-4-E2B-it",           "model_id": "google/gemma-4-E2B-it",               "family": "gemma4",  "loader": "gemma4_processor",  "chat_kwargs": {}},
    {"name": "Gemma-4-E4B-it",           "model_id": "google/gemma-4-E4B-it",               "family": "gemma4",  "loader": "gemma4_processor",  "chat_kwargs": {}},
    {"name": "Llama-3.2-1B-Instruct",    "model_id": "meta-llama/Llama-3.2-1B-Instruct",    "family": "llama",   "loader": "causal",            "chat_kwargs": {}},
    {"name": "Llama-3.2-3B-Instruct",    "model_id": "meta-llama/Llama-3.2-3B-Instruct",    "family": "llama",   "loader": "causal",            "chat_kwargs": {}},
    {"name": "Granite-3.3-2B-Instruct",  "model_id": "ibm-granite/granite-3.3-2b-instruct", "family": "granite", "loader": "causal",            "chat_kwargs": {}},
    {"name": "SmolLM3-3B",               "model_id": "HuggingFaceTB/SmolLM3-3B",             "family": "smollm",  "loader": "causal",            "chat_kwargs": {}},
    {"name": "Phi-4-mini-instruct",      "model_id": "microsoft/Phi-4-mini-instruct",        "family": "phi",     "loader": "causal",            "chat_kwargs": {}},
]

CONDITIONS = {
    "A": {"context_mode": "full",             "verification": True, "repair": False},
    "B": {"context_mode": "pruned",           "verification": True, "repair": False},
    "C": {"context_mode": "pruned_augmented", "verification": True, "repair": False},
    "D": {"context_mode": "pruned_augmented", "verification": True, "repair": True},
}

print(f"DEVICE={DEVICE}, DTYPE={DTYPE}")
print(f"Models={len(MODEL_REGISTRY)}, Conditions={list(CONDITIONS.keys())}")

ROOT: /content/drive/MyDrive/nl2sql_sqlite
RESULTS_DIR: /content/drive/MyDrive/nl2sql_sqlite/results
CACHE_DIR: /content/nl2sql_hf_cache
SPIDER_ROOT: /content/drive/MyDrive/data/spider
Root exists: True
dev.json exists: True
tables.json exists: True
database dir exists: True
DEVICE=cuda, DTYPE=torch.bfloat16
Models=10, Conditions=['A', 'B', 'C', 'D']


In [ ]:
# CELL 4: Log in to Hugging Face when a gated model needs access.
from huggingface_hub import login, whoami

HF_TOKEN = "<YOUR_HUGGING_FACE_TOKEN_GOES_HERE>"

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    try:
        print("Logged into Hugging Face as:", whoami()["name"])
    except Exception:
        print("Logged into Hugging Face.")
else:
    print("No Hugging Face token provided. Public models may still run. Gated models will fail until you log in.")

Logged into Hugging Face as: havietthang


In [ ]:
# CELL 5: Define memory snapshots and cleanup helpers so each model run stays as isolated as possible.
def bytes_to_mb(x: float) -> float:
    return float(x) / (1024 ** 2)

def process_rss_bytes() -> int:
    return psutil.Process(os.getpid()).memory_info().rss

def cuda_allocated_bytes() -> int:
    if not torch.cuda.is_available():
        return 0
    return torch.cuda.memory_allocated()

def cuda_reserved_bytes() -> int:
    if not torch.cuda.is_available():
        return 0
    return torch.cuda.memory_reserved()

def cuda_peak_allocated_bytes() -> int:
    if not torch.cuda.is_available():
        return 0
    return torch.cuda.max_memory_allocated()

def hard_cleanup(model=None, tokenizer=None, processor=None):
    try:
        del model
    except Exception:
        pass
    try:
        del tokenizer
    except Exception:
        pass
    try:
        del processor
    except Exception:
        pass

    gc.collect()

    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass
        try:
            torch.cuda.reset_peak_memory_stats()
        except Exception:
            pass

@dataclass
class MemorySnapshot:
    cpu_rss_mb: float
    gpu_allocated_mb: float
    gpu_reserved_mb: float
    gpu_peak_allocated_mb: float

def take_memory_snapshot() -> MemorySnapshot:
    return MemorySnapshot(
        cpu_rss_mb=bytes_to_mb(process_rss_bytes()),
        gpu_allocated_mb=bytes_to_mb(cuda_allocated_bytes()),
        gpu_reserved_mb=bytes_to_mb(cuda_reserved_bytes()),
        gpu_peak_allocated_mb=bytes_to_mb(cuda_peak_allocated_bytes()),
    )

def snapshot_delta(before: MemorySnapshot, after: MemorySnapshot) -> Dict[str, float]:
    return {
        "cpu_rss_delta_mb": after.cpu_rss_mb - before.cpu_rss_mb,
        "gpu_allocated_delta_mb": after.gpu_allocated_mb - before.gpu_allocated_mb,
        "gpu_reserved_delta_mb": after.gpu_reserved_mb - before.gpu_reserved_mb,
        "gpu_peak_allocated_delta_mb": after.gpu_peak_allocated_mb - before.gpu_peak_allocated_mb,
    }

In [ ]:
# CELL 6: Prepare 4-bit model loading and handle model-specific loading quirks.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=DTYPE,
)

@dataclass
class ModelBundle:
    model_name: str
    model_id: str
    family: str
    loader: str
    model: Any
    tokenizer: Any = None
    processor: Any = None
    load_time_sec: float = 0.0
    load_memory: Optional[Dict[str, float]] = None

def _ensure_chat_template_from_jinja(obj_with_tokenizer: Any, model_id: str):
    tok = None
    if hasattr(obj_with_tokenizer, "tokenizer") and obj_with_tokenizer.tokenizer is not None:
        tok = obj_with_tokenizer.tokenizer
    else:
        tok = obj_with_tokenizer

    if getattr(tok, "chat_template", None):
        return

    try:
        jinja_path = hf_hub_download(
            repo_id=model_id,
            filename="chat_template.jinja",
            cache_dir=str(CACHE_DIR),
        )
        with open(jinja_path, "r", encoding="utf-8") as f:
            tok.chat_template = f.read()
    except Exception:
        pass

def _get_tokenizer_from_bundle(bundle: ModelBundle):
    if bundle.loader == "gemma4_processor":
        return bundle.processor.tokenizer
    return bundle.tokenizer

def _ensure_pad_eos_tokens(bundle: ModelBundle):
    tok = _get_tokenizer_from_bundle(bundle)

    if getattr(tok, "eos_token", None) is None and getattr(tok, "eos_token_id", None) is not None:
        try:
            tok.eos_token = tok.convert_ids_to_tokens(tok.eos_token_id)
        except Exception:
            pass

    if getattr(tok, "pad_token", None) is None:
        if getattr(tok, "eos_token", None) is not None:
            tok.pad_token = tok.eos_token
        elif getattr(tok, "eos_token_id", None) is not None:
            try:
                tok.pad_token = tok.convert_ids_to_tokens(tok.eos_token_id)
            except Exception:
                pass

    if getattr(tok, "pad_token_id", None) is None and getattr(tok, "eos_token_id", None) is not None:
        try:
            tok.pad_token_id = tok.eos_token_id
        except Exception:
            pass

def load_model_bundle(spec: Dict[str, Any]) -> ModelBundle:
    hard_cleanup()

    before = take_memory_snapshot()
    t0 = time.perf_counter()

    common_kwargs = dict(
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=DTYPE,
        trust_remote_code=True,
        cache_dir=str(CACHE_DIR),
    )

    if spec["loader"] == "gemma4_processor":
        processor = AutoProcessor.from_pretrained(
            spec["model_id"],
            trust_remote_code=True,
            cache_dir=str(CACHE_DIR),
        )
        _ensure_chat_template_from_jinja(processor, spec["model_id"])

        model = AutoModelForImageTextToText.from_pretrained(
            spec["model_id"],
            **common_kwargs,
        )

        bundle = ModelBundle(
            model_name=spec["name"],
            model_id=spec["model_id"],
            family=spec["family"],
            loader=spec["loader"],
            model=model,
            processor=processor,
            tokenizer=processor.tokenizer,
        )
        _ensure_pad_eos_tokens(bundle)

    else:
        tokenizer = AutoTokenizer.from_pretrained(
            spec["model_id"],
            trust_remote_code=True,
            cache_dir=str(CACHE_DIR),
        )

        if tokenizer.pad_token is None and tokenizer.eos_token is not None:
            tokenizer.pad_token = tokenizer.eos_token
        if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
            tokenizer.pad_token_id = tokenizer.eos_token_id

        model = AutoModelForCausalLM.from_pretrained(
            spec["model_id"],
            **common_kwargs,
        )

        bundle = ModelBundle(
            model_name=spec["name"],
            model_id=spec["model_id"],
            family=spec["family"],
            loader=spec["loader"],
            model=model,
            tokenizer=tokenizer,
        )
        _ensure_pad_eos_tokens(bundle)

    load_time = time.perf_counter() - t0
    after = take_memory_snapshot()

    bundle.load_time_sec = load_time
    bundle.load_memory = snapshot_delta(before, after)
    return bundle

In [ ]:
# CELL 7: Load the Spider dev set and point each example to its SQLite database.
def assert_spider_layout(spider_root: Path):
    if not spider_root.exists():
        raise FileNotFoundError(
            f"Spider root not found: {spider_root}\n"
            "Set SPIDER_ROOT to the folder containing dev.json, tables.json, and database/."
        )
    if not (spider_root / "dev.json").exists():
        raise FileNotFoundError(f"Missing dev.json under {spider_root}")
    if not (spider_root / "tables.json").exists():
        raise FileNotFoundError(f"Missing tables.json under {spider_root}")
    if not (spider_root / "database").exists():
        raise FileNotFoundError(f"Missing database/ under {spider_root}")

def load_spider_dev(spider_root: Path) -> Tuple[pd.DataFrame, List[Dict[str, Any]]]:
    assert_spider_layout(spider_root)

    with open(spider_root / "dev.json", "r", encoding="utf-8") as f:
        dev = json.load(f)
    with open(spider_root / "tables.json", "r", encoding="utf-8") as f:
        tables = json.load(f)

    rows = []
    for idx, ex in enumerate(dev):
        rows.append({
            "example_id": idx,
            "db_id": ex["db_id"],
            "question": ex["question"],
            "gold_sql": ex["query"],
        })

    return pd.DataFrame(rows), tables

def sqlite_path_for_db(db_id: str) -> Path:
    return SPIDER_ROOT / "database" / db_id / f"{db_id}.sqlite"

def sqlite_connect_safe(sqlite_path: Path, timeout: float = SQLITE_TIMEOUT_SEC):
    conn = sqlite3.connect(str(sqlite_path), timeout=timeout)
    conn.text_factory = lambda b: b.decode("utf-8", errors="replace") if isinstance(b, (bytes, bytearray)) else b
    return conn

def sqlite_table_names(sqlite_path: Path) -> List[str]:
    conn = sqlite_connect_safe(sqlite_path)
    q = "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name;"
    names = [row[0] for row in conn.execute(q).fetchall()]
    conn.close()
    return names

dev_df, spider_tables = load_spider_dev(SPIDER_ROOT)
examples_for_benchmark = dev_df.copy()

print(dev_df.head())
print(f"Spider dev examples: {len(dev_df)}")
print("Example SQLite path:", sqlite_path_for_db(dev_df.iloc[0]["db_id"]))

   example_id           db_id  \
0           0  concert_singer   
1           1  concert_singer   
2           2  concert_singer   
3           3  concert_singer   
4           4  concert_singer   

                                            question  \
0                       How many singers do we have?   
1               What is the total number of singers?   
2  Show name, country, age for all singers ordere...   
3  What are the names, countries, and ages for ev...   
4  What is the average, minimum, and maximum age ...   

                                            gold_sql  
0                        SELECT count(*) FROM singer  
1                        SELECT count(*) FROM singer  
2  SELECT name ,  country ,  age FROM singer ORDE...  
3  SELECT name ,  country ,  age FROM singer ORDE...  
4  SELECT avg(age) ,  min(age) ,  max(age) FROM s...  
Spider dev examples: 1034
Example SQLite path: /content/drive/MyDrive/data/spider/database/concert_singer/concert_singer.sqlite


In [ ]:
# CELL 8: Build schema text for the full, pruned, and augmented context settings.
def normalize_name(text: str) -> List[str]:
    text = str(text).lower().strip()
    text = re.sub(r"[^a-z0-9_ ]+", " ", text)
    tokens = []
    for chunk in text.replace("_", " ").split():
        if chunk:
            tokens.append(chunk)
    return tokens

def build_schema_index(tables_json: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    schema_by_db = {}

    for db in tables_json:
        db_id = db["db_id"]
        table_names = db["table_names_original"]
        col_entries = db["column_names_original"]
        col_types = db["column_types"]
        pk_idxs = set(db["primary_keys"])
        fk_pairs = db["foreign_keys"]

        tables = []
        for t_idx, t_name in enumerate(table_names):
            tables.append({
                "table_id": t_idx,
                "name": t_name,
                "columns": [],
                "primary_keys": [],
                "foreign_keys_out": [],
                "foreign_keys_in": [],
            })

        for c_idx, (t_idx, c_name) in enumerate(col_entries):
            if t_idx == -1:
                continue

            col_obj = {
                "column_id": c_idx,
                "name": c_name,
                "type": col_types[c_idx],
                "is_primary_key": c_idx in pk_idxs,
            }

            tables[t_idx]["columns"].append(col_obj)

            if c_idx in pk_idxs:
                tables[t_idx]["primary_keys"].append(c_name)

        for src_idx, dst_idx in fk_pairs:
            src_t_idx, src_c_name = col_entries[src_idx]
            dst_t_idx, dst_c_name = col_entries[dst_idx]

            if src_t_idx == -1 or dst_t_idx == -1:
                continue

            edge = {
                "from_table": table_names[src_t_idx],
                "from_column": src_c_name,
                "to_table": table_names[dst_t_idx],
                "to_column": dst_c_name,
            }

            tables[src_t_idx]["foreign_keys_out"].append(edge)
            tables[dst_t_idx]["foreign_keys_in"].append(edge)

        schema_by_db[db_id] = {
            "db_id": db_id,
            "tables": tables,
        }

    return schema_by_db

SCHEMA_INDEX = build_schema_index(spider_tables)

def gold_sql_tables_columns(sql_text: str) -> Tuple[set, set]:
    tables = set()
    columns = set()

    try:
        tree = sqlglot.parse_one(sql_text, read="sqlite")

        for node in tree.find_all(exp.Table):
            if node.name:
                tables.add(node.name.lower())

        for node in tree.find_all(exp.Column):
            if node.name:
                columns.add(node.name.lower())

    except Exception:
        pass

    return tables, columns

def table_score(question: str, table_obj: Dict[str, Any]) -> float:
    q_tokens = set(normalize_name(question))
    score = 0.0

    table_tokens = set(normalize_name(table_obj["name"]))
    score += len(q_tokens & table_tokens) * 3.0

    for col in table_obj["columns"]:
        col_tokens = set(normalize_name(col["name"]))
        score += len(q_tokens & col_tokens) * 1.0

    return score

def select_tables_conservative(question: str, schema: Dict[str, Any], top_k: int = TOP_K_TABLES) -> List[Dict[str, Any]]:
    scored = []

    for table in schema["tables"]:
        scored.append((table_score(question, table), table))

    scored.sort(key=lambda x: x[0], reverse=True)

    top = [t for s, t in scored[:top_k] if s > 0]

    if not top:
        return schema["tables"][:]

    selected_names = {t["name"] for t in top}
    all_tables = {t["name"]: t for t in schema["tables"]}

    for t in top[:]:
        for edge in t["foreign_keys_out"] + t["foreign_keys_in"]:
            if edge["from_table"] in all_tables:
                selected_names.add(edge["from_table"])
            if edge["to_table"] in all_tables:
                selected_names.add(edge["to_table"])

    return [all_tables[name] for name in selected_names]

def fetch_sample_values(db_id: str, table_name: str, column_name: str, limit: int = SAMPLE_VALUES_PER_TEXT_COL) -> List[str]:
    samples = []
    sqlite_path = sqlite_path_for_db(db_id)

    try:
        conn = sqlite_connect_safe(sqlite_path)
        q = f'''
            SELECT DISTINCT "{column_name}"
            FROM "{table_name}"
            WHERE "{column_name}" IS NOT NULL
            LIMIT {limit}
        '''
        rows = conn.execute(q).fetchall()
        conn.close()

        for r in rows:
            v = r[0]
            if isinstance(v, bytes):
                v = v.decode("utf-8", errors="replace")
            if isinstance(v, str):
                v = v.replace("\x00", "")
                if len(v) <= 40:
                    samples.append(v)

    except Exception:
        pass

    return samples

def schema_to_text(db_id: str, schema_tables: List[Dict[str, Any]], augmented: bool = False) -> str:
    lines = []

    for table in sorted(schema_tables, key=lambda x: x["name"].lower()):
        lines.append(f'Table: {table["name"]}')

        for col in table["columns"]:
            if augmented:
                col_line = f'- {col["name"]} {str(col["type"]).upper()}'

                if col.get("is_primary_key", False):
                    col_line += ", primary key"

                if str(col["type"]).lower() in {"text", "varchar"}:
                    vals = fetch_sample_values(db_id, table["name"], col["name"])
                    if vals:
                        vals_str = ", ".join([repr(v) for v in vals[:SAMPLE_VALUES_PER_TEXT_COL]])
                        col_line += f", sample values: {vals_str}"

                lines.append(col_line)
            else:
                lines.append(f'- {col["name"]}')

        lines.append("")

    fk_lines = []

    for table in schema_tables:
        for edge in table["foreign_keys_out"]:
            fk_lines.append(f'{edge["from_table"]}.{edge["from_column"]} -> {edge["to_table"]}.{edge["to_column"]}')

    if fk_lines:
        lines.append("Foreign keys:")
        for x in sorted(set(fk_lines)):
            lines.append(x)

    return "\n".join(lines).strip()

def build_schema_context(db_id: str, question: str, gold_sql: Optional[str], context_mode: str) -> Dict[str, Any]:
    schema = SCHEMA_INDEX[db_id]

    full_tables = schema["tables"]
    full_context = schema_to_text(db_id, full_tables, augmented=False)

    if context_mode == "full":
        selected_tables = full_tables
        schema_context = full_context
        augmented = False

    elif context_mode == "pruned":
        selected_tables = select_tables_conservative(question, schema, TOP_K_TABLES)
        schema_context = schema_to_text(db_id, selected_tables, augmented=False)
        augmented = False

    elif context_mode == "pruned_augmented":
        selected_tables = select_tables_conservative(question, schema, TOP_K_TABLES)
        schema_context = schema_to_text(db_id, selected_tables, augmented=True)
        augmented = True

    else:
        raise ValueError(f"Unknown context_mode: {context_mode}")

    gold_tables, gold_columns = gold_sql_tables_columns(gold_sql or "")

    selected_table_names = {t["name"].lower() for t in selected_tables}
    selected_column_names = {c["name"].lower() for t in selected_tables for c in t["columns"]}

    gold_table_recall = int(gold_tables.issubset(selected_table_names)) if gold_tables else 1
    gold_column_recall = int(gold_columns.issubset(selected_column_names)) if gold_columns else 1

    pruning_failure = int(
        (context_mode != "full")
        and (
            not gold_tables.issubset(selected_table_names)
            or not gold_columns.issubset(selected_column_names)
        )
    )

    return {
        "db_id": db_id,
        "context_mode": context_mode,
        "schema_context": schema_context,
        "schema_context_chars": len(schema_context),
        "full_schema_chars": len(full_context),
        "schema_compression_ratio": len(schema_context) / max(len(full_context), 1),
        "selected_tables": [t["name"] for t in selected_tables],
        "gold_table_recall": gold_table_recall,
        "gold_column_recall": gold_column_recall,
        "pruning_failure": pruning_failure,
        "augmented": augmented,
    }

In [ ]:
# CELL 9: Turn each question and schema into a prompt, run generation, and extract the SQL.
SYSTEM_PROMPT = (
    "You are a careful SQL assistant. "
    "Generate exactly one SQLite-compatible SELECT query that answers the user question. "
    "Return only the SQL query. Do not explain. Do not include markdown fences."
)

def build_generation_prompt(question: str, schema_context: str) -> str:
    return (
        "You are given a database schema and a user question.\n"
        "Generate one SQLite-compatible SQL query that answers the question.\n"
        "Return only the SQL query. Do not explain.\n\n"
        f"Schema:\n{schema_context}\n\n"
        f"Question:\n{question}\n\n"
        "SQL:\n"
    )

def _standard_text_messages(prompt: str) -> List[Dict[str, Any]]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

def _gemma4_text_messages(prompt: str) -> List[Dict[str, Any]]:
    return [
        {
            "role": "system",
            "content": [{"type": "text", "text": SYSTEM_PROMPT}],
        },
        {
            "role": "user",
            "content": [{"type": "text", "text": prompt}],
        },
    ]

def apply_chat_template(bundle: ModelBundle, prompt: str, spec: Dict[str, Any]) -> Dict[str, torch.Tensor]:
    if bundle.loader == "gemma4_processor":
        proc = bundle.processor
        messages = _gemma4_text_messages(prompt)

        try:
            model_inputs = proc.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
                **spec.get("chat_kwargs", {}),
            )
        except Exception:
            rendered = proc.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=False,
                **spec.get("chat_kwargs", {}),
            )
            model_inputs = proc(
                text=rendered,
                return_tensors="pt",
            )

    else:
        tok = bundle.tokenizer
        messages = _standard_text_messages(prompt)

        rendered = tok.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
            **spec.get("chat_kwargs", {}),
        )

        model_inputs = tok(
            rendered,
            return_tensors="pt",
        )

    return {k: v.to(bundle.model.device) for k, v in model_inputs.items() if hasattr(v, "to")}

def get_generation_token_ids(bundle: ModelBundle) -> Dict[str, Optional[int]]:
    tok = _get_tokenizer_from_bundle(bundle)

    eos_token_id = getattr(tok, "eos_token_id", None)
    pad_token_id = getattr(tok, "pad_token_id", None)

    if isinstance(eos_token_id, list):
        eos_token_id = eos_token_id[0] if eos_token_id else None
    if isinstance(pad_token_id, list):
        pad_token_id = pad_token_id[0] if pad_token_id else None

    if eos_token_id is None:
        eos_token_id = getattr(bundle.model.config, "eos_token_id", None)
    if isinstance(eos_token_id, list):
        eos_token_id = eos_token_id[0] if eos_token_id else None

    if pad_token_id is None:
        pad_token_id = eos_token_id

    if pad_token_id is None:
        pad_token_id = 0
    if eos_token_id is None:
        eos_token_id = pad_token_id

    return {
        "pad_token_id": int(pad_token_id),
        "eos_token_id": int(eos_token_id),
    }

def get_prompt_length_from_inputs(model_inputs: Dict[str, torch.Tensor]) -> int:
    return int(model_inputs["input_ids"].shape[1])

def decode_generated(bundle: ModelBundle, generated_ids: torch.Tensor, prompt_len: int) -> str:
    gen_only = generated_ids[:, prompt_len:]

    if bundle.loader == "gemma4_processor":
        return bundle.processor.batch_decode(gen_only, skip_special_tokens=True)[0]

    try:
        return bundle.tokenizer.batch_decode(gen_only, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    except TypeError:
        return bundle.tokenizer.batch_decode(gen_only, skip_special_tokens=True)[0]

def strip_reasoning_markup(text: str) -> str:
    text = re.sub(r"<think>.*?</think>", " ", text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r"<response>|</response>", " ", text, flags=re.IGNORECASE)
    return text.strip()

def extract_first_select_sql(raw_text: str) -> Tuple[str, str]:
    if raw_text is None:
        return "", "empty_raw"

    text = strip_reasoning_markup(raw_text)
    text = text.replace("```sql", "```").replace("```SQL", "```")
    text = re.sub(r"^SQL\s*:\s*", "", text.strip(), flags=re.IGNORECASE)

    fenced = re.findall(r"```(.*?)```", text, flags=re.DOTALL)

    if fenced:
        text = fenced[0].strip()

    match = re.search(r"(SELECT\b.*?)(;|$)", text, flags=re.IGNORECASE | re.DOTALL)

    if not match:
        return text.strip(), "no_select_match"

    sql_text = match.group(1).strip()
    sql_text = re.sub(r"\s+", " ", sql_text).strip()

    return sql_text, "ok"

def generate_sql(bundle: ModelBundle, spec: Dict[str, Any], prompt: str) -> Dict[str, Any]:
    before = take_memory_snapshot()
    t0 = time.perf_counter()

    model_inputs = apply_chat_template(bundle, prompt, spec)
    prompt_len = get_prompt_length_from_inputs(model_inputs)
    token_ids = get_generation_token_ids(bundle)

    gen_kwargs = dict(
        **model_inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        use_cache=True,
        pad_token_id=token_ids["pad_token_id"],
        eos_token_id=token_ids["eos_token_id"],
    )

    with torch.no_grad():
        generated = bundle.model.generate(**gen_kwargs)

    latency = time.perf_counter() - t0
    after = take_memory_snapshot()

    raw_text = decode_generated(bundle, generated, prompt_len)
    extracted_sql, extraction_status = extract_first_select_sql(raw_text)

    return {
        "raw_output": raw_text,
        "generated_sql": extracted_sql,
        "extraction_status": extraction_status,
        "generation_latency_sec": latency,
        "runtime_memory_delta": snapshot_delta(before, after),
        "prompt_input_tokens": prompt_len,
        "prompt_input_chars": len(prompt),
    }

In [ ]:
# CELL 10: Check and execute generated SQLite queries safely.
def parse_sql(sql_text: str) -> Tuple[bool, Optional[Any], Optional[str]]:
    if not sql_text or not str(sql_text).strip():
        return False, None, "empty_output"

    if not re.match(r"^\s*SELECT\b", str(sql_text), flags=re.IGNORECASE):
        return False, None, "non_select_query"

    try:
        tree = sqlglot.parse_one(sql_text, read="sqlite")
        return True, tree, None

    except Exception:
        return False, None, "parse_error"

def referenced_tables(sql_tree: Any) -> List[str]:
    tables = []

    try:
        for node in sql_tree.find_all(exp.Table):
            if node.name:
                tables.append(node.name.lower())
    except Exception:
        pass

    return sorted(set(tables))

def schema_table_exists(db_id: str, table_names: List[str]) -> bool:
    schema = SCHEMA_INDEX[db_id]
    known = {t["name"].lower() for t in schema["tables"]}
    return all(t in known for t in table_names)

def looks_like_unsafe_cartesian_product(sql_text: str) -> bool:
    if not sql_text:
        return False

    s = re.sub(r"\s+", " ", sql_text.strip().lower())

    has_comma_from = bool(re.search(r"\bfrom\b[^;]*,", s))
    has_join = " join " in s
    has_where = " where " in s
    has_limit = " limit " in s

    return has_comma_from and not has_join and not has_where and not has_limit

def execute_sqlite_sql(db_id: str, sql_text: str, timeout_sec: float = SQLITE_TIMEOUT_SEC) -> Tuple[bool, Optional[pd.DataFrame], Optional[str]]:
    if looks_like_unsafe_cartesian_product(sql_text):
        return False, None, "unsafe_cartesian_product_guard"

    sqlite_path = sqlite_path_for_db(db_id)
    conn = None
    start = time.perf_counter()

    try:
        conn = sqlite_connect_safe(sqlite_path, timeout=timeout_sec)

        def progress_handler():
            if time.perf_counter() - start > timeout_sec:
                return 1
            return 0

        conn.set_progress_handler(progress_handler, SQLITE_PROGRESS_STEPS)
        df = pd.read_sql_query(sql_text, conn)
        conn.close()
        return True, df, None

    except Exception as e:
        try:
            if conn is not None:
                conn.close()
        except Exception:
            pass

        err = repr(e)
        lowered = err.lower()
        if "interrupted" in lowered:
            return False, None, "execution_timeout"
        return False, None, err

def verify_sql(db_id: str, sql_text: str) -> Dict[str, Any]:
    t0 = time.perf_counter()

    ok, tree, parse_label = parse_sql(sql_text)

    if not ok:
        return {
            "verification_status": parse_label,
            "verification_error_message": parse_label,
            "verification_latency_sec": time.perf_counter() - t0,
        }

    ref_tables = referenced_tables(tree)

    if ref_tables and not schema_table_exists(db_id, ref_tables):
        return {
            "verification_status": "schema_error",
            "verification_error_message": f"Unknown referenced table(s): {ref_tables}",
            "verification_latency_sec": time.perf_counter() - t0,
        }

    if looks_like_unsafe_cartesian_product(sql_text):
        return {
            "verification_status": "unsafe_cartesian_product",
            "verification_error_message": "Generated SQL uses comma-style multi-table FROM without JOIN/WHERE/LIMIT.",
            "verification_latency_sec": time.perf_counter() - t0,
        }

    exec_ok, _, exec_err = execute_sqlite_sql(db_id, sql_text)

    if not exec_ok:
        lowered = (exec_err or "").lower()

        if exec_err == "execution_timeout":
            status = "execution_timeout"
        elif exec_err == "unsafe_cartesian_product_guard":
            status = "unsafe_cartesian_product"
        elif any(x in lowered for x in ["no such table", "no such column", "ambiguous column", "syntax error"]):
            status = "schema_error"
        else:
            status = "execution_error"

        return {
            "verification_status": status,
            "verification_error_message": exec_err,
            "verification_latency_sec": time.perf_counter() - t0,
        }

    return {
        "verification_status": "success",
        "verification_error_message": None,
        "verification_latency_sec": time.perf_counter() - t0,
    }

def has_order_by(sql_text: str) -> bool:
    try:
        tree = sqlglot.parse_one(sql_text, read="sqlite")
        return any(True for _ in tree.find_all(exp.Order))
    except Exception:
        return False

def canonicalize_df(df: pd.DataFrame, preserve_order: bool) -> pd.DataFrame:
    if df is None:
        return df

    out = df.copy()
    out.columns = [str(c) for c in out.columns]

    for c in out.columns:
        out[c] = out[c].apply(lambda x: None if pd.isna(x) else x)

    if preserve_order:
        return out.reset_index(drop=True)

    sort_cols = list(out.columns)

    try:
        return out.astype(str).sort_values(by=sort_cols, kind="mergesort").reset_index(drop=True)
    except Exception:
        return out.astype(str).reset_index(drop=True)

def execution_match(db_id: str, pred_sql: str, gold_sql: str) -> Dict[str, Any]:
    gold_ok, gold_df, gold_err = execute_sqlite_sql(db_id, gold_sql)
    pred_ok, pred_df, pred_err = execute_sqlite_sql(db_id, pred_sql)

    if not gold_ok:
        return {
            "gold_exec_ok": False,
            "pred_exec_ok": pred_ok,
            "execution_correct": None,
            "gold_exec_error": gold_err,
            "pred_exec_error": pred_err,
        }

    if not pred_ok:
        return {
            "gold_exec_ok": True,
            "pred_exec_ok": False,
            "execution_correct": 0,
            "gold_exec_error": None,
            "pred_exec_error": pred_err,
        }

    preserve_order = has_order_by(gold_sql)

    gold_can = canonicalize_df(gold_df, preserve_order=preserve_order)
    pred_can = canonicalize_df(pred_df, preserve_order=preserve_order)

    same = int(gold_can.equals(pred_can))

    return {
        "gold_exec_ok": True,
        "pred_exec_ok": True,
        "execution_correct": same,
        "gold_exec_error": None,
        "pred_exec_error": None,
    }

In [ ]:
# CELL 11: Build the one-step repair prompt for queries that fail verification.
def build_repair_prompt(question: str, schema_context: str, failed_sql: str, error_message: str) -> str:
    return (
        "The SQL query below failed.\n\n"
        f"Schema:\n{schema_context}\n\n"
        f"Question:\n{question}\n\n"
        f"Failed SQL:\n{failed_sql}\n\n"
        f"Error:\n{error_message}\n\n"
        "Return one corrected SQLite-compatible SELECT query.\n"
        "Return only the SQL query.\n"
    )

def maybe_repair(
    bundle: ModelBundle,
    spec: Dict[str, Any],
    question: str,
    schema_context: str,
    failed_sql: str,
    verification_status: str,
    verification_error_message: str,
) -> Dict[str, Any]:
    repair_prompt = build_repair_prompt(
        question,
        schema_context,
        failed_sql,
        verification_error_message or verification_status,
    )

    repair_gen = generate_sql(bundle, spec, repair_prompt)

    return {
        "repair_prompt": repair_prompt,
        "repair_raw_output": repair_gen["raw_output"],
        "repair_sql": repair_gen["generated_sql"],
        "repair_extraction_status": repair_gen["extraction_status"],
        "repair_latency_sec": repair_gen["generation_latency_sec"],
        "repair_runtime_memory_delta": repair_gen["runtime_memory_delta"],
    }

In [ ]:
# CELL 12: Run one model on one example under one condition and collect all logged fields.
def run_single_example(
    bundle: ModelBundle,
    spec: Dict[str, Any],
    example_row: pd.Series,
    condition_name: str,
) -> Dict[str, Any]:
    cond = CONDITIONS[condition_name]

    db_id = example_row["db_id"]
    question = example_row["question"]
    gold_sql = example_row["gold_sql"]

    context_info = build_schema_context(
        db_id=db_id,
        question=question,
        gold_sql=gold_sql,
        context_mode=cond["context_mode"],
    )

    prompt = build_generation_prompt(
        question=question,
        schema_context=context_info["schema_context"],
    )

    gen_info = generate_sql(bundle, spec, prompt)
    ver_info = verify_sql(db_id=db_id, sql_text=gen_info["generated_sql"])

    final_sql = gen_info["generated_sql"]

    repair_info = {
        "repair_attempted": 0,
        "repair_sql": None,
        "repair_latency_sec": 0.0,
        "repair_raw_output": None,
        "repair_extraction_status": None,
        "repair_runtime_memory_delta": None,
    }

    if cond["repair"] and ver_info["verification_status"] != "success":
        repair_info["repair_attempted"] = 1

        tmp = maybe_repair(
            bundle=bundle,
            spec=spec,
            question=question,
            schema_context=context_info["schema_context"],
            failed_sql=gen_info["generated_sql"],
            verification_status=ver_info["verification_status"],
            verification_error_message=ver_info["verification_error_message"],
        )

        repair_ver = verify_sql(db_id=db_id, sql_text=tmp["repair_sql"])

        if repair_ver["verification_status"] == "success":
            final_sql = tmp["repair_sql"]
            ver_info = repair_ver

        repair_info.update(tmp)

    exec_info = execution_match(
        db_id=db_id,
        pred_sql=final_sql,
        gold_sql=gold_sql,
    )

    row = {
        "example_id": int(example_row["example_id"]),
        "db_id": db_id,
        "question": question,
        "gold_sql": gold_sql,
        "model_name": bundle.model_name,
        "model_id": bundle.model_id,
        "condition": condition_name,
        "context_mode": cond["context_mode"],
        "verification_enabled": int(cond["verification"]),
        "repair_enabled": int(cond["repair"]),
        "schema_context": context_info["schema_context"],
        "schema_context_chars": context_info["schema_context_chars"],
        "full_schema_chars": context_info["full_schema_chars"],
        "schema_compression_ratio": context_info["schema_compression_ratio"],
        "selected_tables": json.dumps(context_info["selected_tables"], ensure_ascii=False),
        "gold_table_recall": context_info["gold_table_recall"],
        "gold_column_recall": context_info["gold_column_recall"],
        "pruning_failure": context_info["pruning_failure"],
        "prompt_input_chars": gen_info["prompt_input_chars"],
        "prompt_input_tokens": gen_info["prompt_input_tokens"],
        "raw_output": gen_info["raw_output"],
        "generated_sql": gen_info["generated_sql"],
        "final_sql": final_sql,
        "extraction_status": gen_info["extraction_status"],
        "verification_status": ver_info["verification_status"],
        "verification_error_message": ver_info["verification_error_message"],
        "verification_latency_sec": ver_info["verification_latency_sec"],
        "generation_latency_sec": gen_info["generation_latency_sec"],
        "repair_attempted": repair_info["repair_attempted"],
        "repair_sql": repair_info["repair_sql"],
        "repair_raw_output": repair_info["repair_raw_output"],
        "repair_extraction_status": repair_info["repair_extraction_status"],
        "repair_latency_sec": repair_info["repair_latency_sec"],
        "gold_exec_ok": exec_info["gold_exec_ok"],
        "pred_exec_ok": exec_info["pred_exec_ok"],
        "execution_correct": exec_info["execution_correct"],
        "gold_exec_error": exec_info["gold_exec_error"],
        "pred_exec_error": exec_info["pred_exec_error"],
        "model_load_time_sec": bundle.load_time_sec,
        "model_load_cpu_delta_mb": bundle.load_memory["cpu_rss_delta_mb"],
        "model_load_gpu_allocated_delta_mb": bundle.load_memory["gpu_allocated_delta_mb"],
        "model_load_gpu_reserved_delta_mb": bundle.load_memory["gpu_reserved_delta_mb"],
        "runtime_cpu_delta_mb": gen_info["runtime_memory_delta"]["cpu_rss_delta_mb"],
        "runtime_gpu_allocated_delta_mb": gen_info["runtime_memory_delta"]["gpu_allocated_delta_mb"],
        "runtime_gpu_reserved_delta_mb": gen_info["runtime_memory_delta"]["gpu_reserved_delta_mb"],
        "runtime_gpu_peak_delta_mb": gen_info["runtime_memory_delta"]["gpu_peak_allocated_delta_mb"],
        "total_system_cpu_rss_mb": take_memory_snapshot().cpu_rss_mb,
        "fatal_error": None,
    }

    return row

In [ ]:
# CELL 13: Save results to JSONL files and summarize them into benchmark metrics.
def append_jsonl(path: Path, row: Dict[str, Any]):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

def read_jsonl(path: Path) -> pd.DataFrame:
    rows = []

    if not path.exists():
        return pd.DataFrame()

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                rows.append(json.loads(line))
            except Exception:
                pass

    return pd.DataFrame(rows)

def usable_completed_pairs(df: pd.DataFrame) -> set:
    if df.empty:
        return set()
    if not {"example_id", "condition", "fatal_error"}.issubset(df.columns):
        return set()
    tmp = df[df["fatal_error"].isna()].copy()
    return set(zip(tmp["example_id"].astype(int), tmp["condition"].astype(str)))

def aggregate_main_metrics(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()

    tmp = df.copy()

    if "fatal_error" not in tmp.columns:
        tmp["fatal_error"] = None

    tmp = tmp[tmp["fatal_error"].isna()].copy()

    if tmp.empty:
        return pd.DataFrame()

    tmp["is_executable"] = (tmp["verification_status"] == "success").astype(int)
    tmp["exact_match"] = (
        tmp["final_sql"].fillna("").str.strip().str.lower()
        == tmp["gold_sql"].fillna("").str.strip().str.lower()
    ).astype(int)

    grouped = (
        tmp.groupby(["model_name", "condition"], dropna=False)
        .agg(
            n_examples=("example_id", "count"),
            execution_accuracy=("execution_correct", "mean"),
            executable_query_rate=("is_executable", "mean"),
            exact_match_accuracy=("exact_match", "mean"),
            median_generation_latency_sec=("generation_latency_sec", "median"),
            mean_generation_latency_sec=("generation_latency_sec", "mean"),
            p95_generation_latency_sec=("generation_latency_sec", lambda x: float(np.percentile(x, 95))),
            mean_prompt_input_chars=("prompt_input_chars", "mean"),
            mean_schema_context_chars=("schema_context_chars", "mean"),
            mean_schema_compression_ratio=("schema_compression_ratio", "mean"),
            gold_table_recall=("gold_table_recall", "mean"),
            gold_column_recall=("gold_column_recall", "mean"),
            pruning_failure_rate=("pruning_failure", "mean"),
            mean_model_load_gpu_reserved_delta_mb=("model_load_gpu_reserved_delta_mb", "mean"),
            mean_runtime_gpu_peak_delta_mb=("runtime_gpu_peak_delta_mb", "mean"),
            mean_runtime_cpu_delta_mb=("runtime_cpu_delta_mb", "mean"),
            repair_attempt_rate=("repair_attempted", "mean"),
            mean_repair_latency_sec=("repair_latency_sec", "mean"),
        )
        .reset_index()
    )

    return grouped

def error_breakdown(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()

    tmp = df.copy()

    if "fatal_error" not in tmp.columns:
        tmp["fatal_error"] = None

    tmp = tmp[tmp["fatal_error"].isna()].copy()

    if tmp.empty:
        return pd.DataFrame()

    out = (
        tmp.groupby(["model_name", "condition", "verification_status"])
        .size()
        .reset_index(name="count")
    )

    totals = tmp.groupby(["model_name", "condition"]).size().reset_index(name="total")
    out = out.merge(totals, on=["model_name", "condition"], how="left")
    out["rate"] = out["count"] / out["total"]

    return out

In [ ]:
# CELL 14: Run a small smoke test before spending compute on the full benchmark.
SMOKE_N = 5
SMOKE_MODEL_NAME = "Qwen3-0.6B"
SMOKE_CONDITIONS = ["A", "B", "C", "D"]

smoke_spec = next(x for x in MODEL_REGISTRY if x["name"] == SMOKE_MODEL_NAME)
bundle = load_model_bundle(smoke_spec)

smoke_out = RESULTS_DIR / f"smoke_{SMOKE_MODEL_NAME.replace('/', '_')}.jsonl"

if smoke_out.exists():
    smoke_out.unlink()

subset = examples_for_benchmark.head(SMOKE_N).copy()

for _, row in tqdm(subset.iterrows(), total=len(subset), desc=f"Smoke test: {SMOKE_MODEL_NAME}"):
    for cond in SMOKE_CONDITIONS:
        result_row = run_single_example(bundle, smoke_spec, row, cond)
        append_jsonl(smoke_out, result_row)

smoke_df = read_jsonl(smoke_out)

display(smoke_df.head(2))
display(aggregate_main_metrics(smoke_df))
display(error_breakdown(smoke_df))

hard_cleanup(bundle.model, bundle.tokenizer, bundle.processor)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Smoke test: Qwen3-0.6B:   0%|          | 0/5 [00:00<?, ?it/s]

,example_id,db_id,question,gold_sql,model_name,model_id,condition,context_mode,verification_enabled,repair_enabled,...,model_load_time_sec,model_load_cpu_delta_mb,model_load_gpu_allocated_delta_mb,model_load_gpu_reserved_delta_mb,runtime_cpu_delta_mb,runtime_gpu_allocated_delta_mb,runtime_gpu_reserved_delta_mb,runtime_gpu_peak_delta_mb,total_system_cpu_rss_mb,fatal_error
0,0,concert_singer,How many singers do we have?,SELECT count(*) FROM singer,Qwen3-0.6B,Qwen/Qwen3-0.6B,A,full,1,0,...,1.930405,0.136719,515.09668,1044.0,0.105469,0.005859,34.0,0.0,2604.410156,None
1,0,concert_singer,How many singers do we have?,SELECT count(*) FROM singer,Qwen3-0.6B,Qwen/Qwen3-0.6B,B,pruned,1,0,...,1.930405,0.136719,515.09668,1044.0,0.000000,0.005859,0.0,0.0,2604.410156,None


,model_name,condition,n_examples,execution_accuracy,executable_query_rate,exact_match_accuracy,median_generation_latency_sec,mean_generation_latency_sec,p95_generation_latency_sec,mean_prompt_input_chars,mean_schema_context_chars,mean_schema_compression_ratio,gold_table_recall,gold_column_recall,pruning_failure_rate,mean_model_load_gpu_reserved_delta_mb,mean_runtime_gpu_peak_delta_mb,mean_runtime_cpu_delta_mb,repair_attempt_rate,mean_repair_latency_sec
0,Qwen3-0.6B,A,5,0.2,1.0,0.2,1.729706,1.520353,2.389725,708.6,455.0,1.000000,1.0,1.0,0.0,1044.0,0.0,0.021875,0.0,0.0
1,Qwen3-0.6B,B,5,0.2,1.0,0.2,1.656907,1.510134,2.552443,651.2,397.6,0.873846,1.0,1.0,0.0,1044.0,0.0,0.000000,0.0,0.0
2,Qwen3-0.6B,C,5,0.0,1.0,0.2,1.144178,1.313613,2.417547,1249.6,996.0,2.189011,1.0,1.0,0.0,1044.0,0.0,0.009375,0.0,0.0
3,Qwen3-0.6B,D,5,0.0,1.0,0.2,1.144756,1.331925,2.400930,1249.6,996.0,2.189011,1.0,1.0,0.0,1044.0,0.0,0.000000,0.0,0.0


,model_name,condition,verification_status,count,total,rate
0,Qwen3-0.6B,A,success,5,5,1.0
1,Qwen3-0.6B,B,success,5,5,1.0
2,Qwen3-0.6B,C,success,5,5,1.0
3,Qwen3-0.6B,D,success,5,5,1.0


In [ ]:
# CELL 15: Define the resumable benchmark runner used by the full model runs.
def run_model_benchmark(
    model_spec: Dict[str, Any],
    examples_df: pd.DataFrame,
    conditions: List[str],
    out_jsonl: Path,
    start_idx: int = 0,
    limit: Optional[int] = None,
):
    bundle = load_model_bundle(model_spec)

    if out_jsonl.exists():
        done_df = read_jsonl(out_jsonl)
        done_keys = usable_completed_pairs(done_df)
    else:
        done_keys = set()

    work_df = examples_df.copy().iloc[start_idx:]

    if limit is not None:
        work_df = work_df.head(limit)

    all_tasks = []
    for _, row in work_df.iterrows():
        for cond in conditions:
            key = (int(row["example_id"]), cond)
            if key not in done_keys:
                all_tasks.append((row, cond, key))

    total_expected = len(examples_df) * len(conditions)
    already_completed = len(done_keys)
    remaining = len(all_tasks)

    print("Output file:", out_jsonl)
    print("Total expected tasks:", total_expected)
    print("Already completed tasks for this model/run:", already_completed)
    print("Remaining tasks this run:", remaining)

    pbar = tqdm(total=total_expected, initial=min(already_completed, total_expected), desc=f"Running {model_spec['name']}", unit="task")

    for row, cond, key in all_tasks:
        try:
            pbar.set_postfix({"example": int(row["example_id"]), "condition": cond})
            result_row = run_single_example(bundle, model_spec, row, cond)

        except Exception as e:
            result_row = {
                "example_id": int(row["example_id"]),
                "db_id": row["db_id"],
                "question": row["question"],
                "gold_sql": row["gold_sql"],
                "model_name": model_spec["name"],
                "model_id": model_spec["model_id"],
                "condition": cond,
                "fatal_error": repr(e),
            }

        append_jsonl(out_jsonl, result_row)
        pbar.update(1)

    pbar.close()
    hard_cleanup(bundle.model, bundle.tokenizer, bundle.processor)

In [ ]:
# CELL 16: Run one complete model first to verify the full pipeline.
FULL_MODEL_NAME = "Qwen3-0.6B"
full_spec = next(x for x in MODEL_REGISTRY if x["name"] == FULL_MODEL_NAME)

out_path = RESULTS_DIR / f"{FULL_MODEL_NAME.replace('/', '_')}_spider_dev.jsonl"

print("Running model:", FULL_MODEL_NAME)
print("Saving to:", out_path)

run_model_benchmark(
    model_spec=full_spec,
    examples_df=examples_for_benchmark,
    conditions=["A", "B", "C", "D"],
    out_jsonl=out_path,
    start_idx=0,
    limit=None,
)

full_df = read_jsonl(out_path)
main_metrics_df = aggregate_main_metrics(full_df)
errors_df = error_breakdown(full_df)

main_metrics_df.to_csv(RESULTS_DIR / f"{FULL_MODEL_NAME}_main_metrics.csv", index=False)
errors_df.to_csv(RESULTS_DIR / f"{FULL_MODEL_NAME}_error_breakdown.csv", index=False)

print("Saved rows:", len(full_df))
display(full_df[full_df["fatal_error"].isna()]["condition"].value_counts().sort_index())
display(full_df.tail(10)[["example_id", "condition", "verification_status", "execution_correct", "fatal_error"]])
display(main_metrics_df)
display(errors_df.head(20))

Running model: Qwen3-0.6B
Saving to: /content/drive/MyDrive/nl2sql_sqlite/results/Qwen3-0.6B_spider_dev.jsonl


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Output file: /content/drive/MyDrive/nl2sql_sqlite/results/Qwen3-0.6B_spider_dev.jsonl
Total expected tasks: 4136
Already completed tasks for this model/run: 4130
Remaining tasks this run: 6


Running Qwen3-0.6B: 100%|#########9| 4130/4136 [00:00<?, ?task/s]

Saved rows: 4207


,count
condition,
A,1032
B,1032
C,1033
D,1033


,example_id,condition,verification_status,execution_correct,fatal_error
4197,400,A,NaN,NaN,ValueError('The truth value of a Series is amb...
4198,400,B,NaN,NaN,ValueError('The truth value of a Series is amb...
4199,541,A,NaN,NaN,ValueError('The truth value of a Series is amb...
4200,541,B,NaN,NaN,ValueError('The truth value of a Series is amb...
4201,392,C,NaN,NaN,ValueError('The truth value of a Series is amb...
4202,392,D,NaN,NaN,ValueError('The truth value of a Series is amb...
4203,400,A,NaN,NaN,ValueError('The truth value of a Series is amb...
4204,400,B,NaN,NaN,ValueError('The truth value of a Series is amb...
4205,541,A,NaN,NaN,ValueError('The truth value of a Series is amb...
4206,541,B,NaN,NaN,ValueError('The truth value of a Series is amb...


,model_name,condition,n_examples,execution_accuracy,executable_query_rate,exact_match_accuracy,median_generation_latency_sec,mean_generation_latency_sec,p95_generation_latency_sec,mean_prompt_input_chars,mean_schema_context_chars,mean_schema_compression_ratio,gold_table_recall,gold_column_recall,pruning_failure_rate,mean_model_load_gpu_reserved_delta_mb,mean_runtime_gpu_peak_delta_mb,mean_runtime_cpu_delta_mb,repair_attempt_rate,mean_repair_latency_sec
0,Qwen3-0.6B,A,1032,0.138700,0.699612,0.060078,1.614218,3.610133,20.282313,844.530039,584.652132,1.000000,1.000000,0.793605,0.000000,1044.0,0.0,0.000079,0.000000,0.000000
1,Qwen3-0.6B,B,1032,0.139535,0.722868,0.056202,1.601228,3.504147,20.251116,753.778101,493.900194,0.887314,0.978682,0.785853,0.220930,1044.0,0.0,-0.043801,0.000000,0.000000
2,Qwen3-0.6B,C,1033,0.134560,0.722168,0.050339,1.552007,3.323121,20.206631,1466.355276,1206.371733,2.190465,0.978703,0.786060,0.220716,1044.0,0.0,0.000125,0.000000,0.000000
3,Qwen3-0.6B,D,1033,0.134560,0.733785,0.050339,1.522983,3.320166,20.235918,1466.355276,1206.371733,2.190465,0.978703,0.786060,0.220716,1044.0,0.0,0.000045,0.277832,1.877456


,model_name,condition,verification_status,count,total,rate
0,Qwen3-0.6B,A,execution_error,11,1032,0.010659
1,Qwen3-0.6B,A,parse_error,80,1032,0.077519
2,Qwen3-0.6B,A,schema_error,210,1032,0.203488
3,Qwen3-0.6B,A,success,722,1032,0.699612
4,Qwen3-0.6B,A,unsafe_cartesian_product,9,1032,0.008721
5,Qwen3-0.6B,B,execution_error,12,1032,0.011628
6,Qwen3-0.6B,B,parse_error,76,1032,0.073643
7,Qwen3-0.6B,B,schema_error,196,1032,0.189922
8,Qwen3-0.6B,B,success,746,1032,0.722868
9,Qwen3-0.6B,B,unsafe_cartesian_product,2,1032,0.001938


In [ ]:
# CELL 17: Run the remaining models one by one while skipping completed results.

EXPECTED_PAIRS_PER_MODEL = len(examples_for_benchmark) * len(CONDITIONS)

original_load_model_bundle = load_model_bundle

def load_model_bundle(spec: Dict[str, Any]) -> ModelBundle:
    if spec["name"] != "Phi-4-mini-instruct":
        return original_load_model_bundle(spec)

    print("Using Phi-specific loader with trust_remote_code=False to avoid remote-code LossKwargs error.")

    hard_cleanup()

    before = take_memory_snapshot()
    t0 = time.perf_counter()

    common_kwargs = dict(
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=DTYPE,
        trust_remote_code=False,
        cache_dir=str(CACHE_DIR),
    )

    tokenizer = AutoTokenizer.from_pretrained(
        spec["model_id"],
        trust_remote_code=False,
        cache_dir=str(CACHE_DIR),
    )

    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token
    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    model = AutoModelForCausalLM.from_pretrained(
        spec["model_id"],
        **common_kwargs,
    )

    bundle = ModelBundle(
        model_name=spec["name"],
        model_id=spec["model_id"],
        family=spec["family"],
        loader=spec["loader"],
        model=model,
        tokenizer=tokenizer,
    )

    _ensure_pad_eos_tokens(bundle)

    load_time = time.perf_counter() - t0
    after = take_memory_snapshot()

    bundle.load_time_sec = load_time
    bundle.load_memory = snapshot_delta(before, after)

    return bundle


print("=" * 80)
print("Full 10-model benchmark launcher")
print("Expected example-condition pairs per model:", EXPECTED_PAIRS_PER_MODEL)
print("=" * 80)

for spec in MODEL_REGISTRY:
    model_name = spec["name"]
    out_path = RESULTS_DIR / f"{model_name.replace('/', '_')}_spider_dev.jsonl"

    print("\n" + "=" * 80)
    print("Model:", model_name)
    print("Output file:", out_path)

    if out_path.exists():
        existing_df = read_jsonl(out_path)
        done_keys = usable_completed_pairs(existing_df)
        completed_pairs = len(done_keys)
    else:
        existing_df = pd.DataFrame()
        completed_pairs = 0

    print(f"Completed pairs: {completed_pairs} / {EXPECTED_PAIRS_PER_MODEL}")

    if completed_pairs >= EXPECTED_PAIRS_PER_MODEL:
        print(f"Skipping {model_name} because it is already complete.")
        continue

    failure_path = RESULTS_DIR / f"{model_name.replace('/', '_')}_model_level_failure.json"

    if failure_path.exists():
        failure_path.unlink()
        print("Deleted old model-level failure report:", failure_path)

    try:
        run_model_benchmark(
            model_spec=spec,
            examples_df=examples_for_benchmark,
            conditions=["A", "B", "C", "D"],
            out_jsonl=out_path,
            start_idx=0,
            limit=None,
        )
    except Exception as e:
        print(f"FAILED to run {model_name}:")
        print(repr(e))

        with open(failure_path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "model_name": model_name,
                    "model_id": spec["model_id"],
                    "error": repr(e),
                    "time": time.strftime("%Y-%m-%d %H:%M:%S"),
                },
                f,
                indent=2,
                ensure_ascii=False,
            )
        print("Saved model-level failure report to:", failure_path)
        hard_cleanup()

print("Full launcher finished.")

Full 10-model benchmark launcher
Expected example-condition pairs per model: 4136

Model: Qwen3-0.6B
Output file: /content/drive/MyDrive/nl2sql_sqlite/results/Qwen3-0.6B_spider_dev.jsonl
Completed pairs: 4130 / 4136


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Output file: /content/drive/MyDrive/nl2sql_sqlite/results/Qwen3-0.6B_spider_dev.jsonl
Total expected tasks: 4136
Already completed tasks for this model/run: 4130
Remaining tasks this run: 6


Running Qwen3-0.6B: 100%|#########9| 4130/4136 [00:00<?, ?task/s]


Model: Qwen3-1.7B
Output file: /content/drive/MyDrive/nl2sql_sqlite/results/Qwen3-1.7B_spider_dev.jsonl
Completed pairs: 4116 / 4136


...

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

Output file: /content/drive/MyDrive/nl2sql_sqlite/results/Phi-4-mini-instruct_spider_dev.jsonl
Total expected tasks: 4136
Already completed tasks for this model/run: 0
Remaining tasks this run: 4136


Running Phi-4-mini-instruct:   0%|          | 0/4136 [00:00<?, ?task/s]

Full launcher finished.


In [ ]:
# CELL 18: Merge all saved model outputs and export the final result tables.
all_rows = []
status_rows = []

for spec in MODEL_REGISTRY:
    model_name = spec["name"]
    path = RESULTS_DIR / f"{model_name.replace('/', '_')}_spider_dev.jsonl"

    if path.exists():
        df = read_jsonl(path)
        rows = len(df)
        completed_pairs = len(usable_completed_pairs(df))
        if not df.empty:
            all_rows.append(df)
    else:
        rows = 0
        completed_pairs = 0

    status_rows.append({
        "model_name": model_name,
        "model_id": spec["model_id"],
        "file_exists": path.exists(),
        "rows": rows,
        "completed_pairs": completed_pairs,
        "expected_pairs": EXPECTED_PAIRS_PER_MODEL,
        "complete": completed_pairs >= EXPECTED_PAIRS_PER_MODEL,
    })

all_df = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
model_file_status_df = pd.DataFrame(status_rows)

if not all_df.empty:
    all_df.to_parquet(RESULTS_DIR / "all_model_outputs.parquet", index=False)
    all_df.to_csv(RESULTS_DIR / "all_model_outputs.csv", index=False)

all_main_metrics = aggregate_main_metrics(all_df)
all_error_breakdown = error_breakdown(all_df)

all_main_metrics.to_csv(RESULTS_DIR / "all_models_main_metrics.csv", index=False)
all_error_breakdown.to_csv(RESULTS_DIR / "all_models_error_breakdown.csv", index=False)
model_file_status_df.to_csv(RESULTS_DIR / "model_file_status.csv", index=False)

print("Merged rows:", len(all_df))
print("Models with output:", model_file_status_df["file_exists"].sum())
print("\nModel file status:")
display(model_file_status_df)

print("Main metrics:")
display(all_main_metrics.sort_values(["condition", "execution_accuracy"], ascending=[True, False]))

print("Error breakdown:")
display(all_error_breakdown.head(100))

Merged rows: 41735
Models with output: 10

Model file status:


,model_name,model_id,file_exists,rows,completed_pairs,expected_pairs,complete
0,Qwen3-0.6B,Qwen/Qwen3-0.6B,True,4213,4130,4136,False
1,Qwen3-1.7B,Qwen/Qwen3-1.7B,True,4256,4116,4136,False
2,Qwen3-4B-Instruct-2507,Qwen/Qwen3-4B-Instruct-2507,True,4184,4128,4136,False
3,Gemma-4-E2B-it,google/gemma-4-E2B-it,True,4183,4127,4136,False
4,Gemma-4-E4B-it,google/gemma-4-E4B-it,True,4160,4128,4136,False
5,Llama-3.2-1B-Instruct,meta-llama/Llama-3.2-1B-Instruct,True,4158,4130,4136,False
6,Llama-3.2-3B-Instruct,meta-llama/Llama-3.2-3B-Instruct,True,4160,4128,4136,False
7,Granite-3.3-2B-Instruct,ibm-granite/granite-3.3-2b-instruct,True,4144,4132,4136,False
8,SmolLM3-3B,HuggingFaceTB/SmolLM3-3B,True,4142,4132,4136,False
9,Phi-4-mini-instruct,microsoft/Phi-4-mini-instruct,True,4135,4129,4136,False


Main metrics:


,model_name,condition,n_examples,execution_accuracy,executable_query_rate,exact_match_accuracy,median_generation_latency_sec,mean_generation_latency_sec,p95_generation_latency_sec,mean_prompt_input_chars,mean_schema_context_chars,mean_schema_compression_ratio,gold_table_recall,gold_column_recall,pruning_failure_rate,mean_model_load_gpu_reserved_delta_mb,mean_runtime_gpu_peak_delta_mb,mean_runtime_cpu_delta_mb,repair_attempt_rate,mean_repair_latency_sec
4,Gemma-4-E4B-it,A,1032,0.525194,0.937984,0.107558,6.705102,7.510141,14.304896,842.904070,583.103682,1.000000,1.000000,0.793605,0.000000,8620.0,0.089667,-0.002021,0.000000,0.000000
32,Qwen3-4B-Instruct-2507,A,1032,0.450581,0.966085,0.074612,4.053564,4.496329,8.244584,842.904070,583.103682,1.000000,1.000000,0.793605,0.000000,2228.0,0.054429,0.000958,0.000000,0.000000
0,Gemma-4-E2B-it,A,1033,0.411423,0.891578,0.060987,7.119118,8.610090,31.575623,844.225557,584.343659,1.000000,1.000000,0.793804,0.000000,6146.0,0.111939,-0.021441,0.000000,0.000000
28,Qwen3-1.7B,A,1029,0.315841,0.848397,0.081633,2.471303,3.246119,7.190730,843.167153,583.337221,1.000000,1.000000,0.793003,0.000000,2754.0,0.000000,-0.000080,0.000000,0.000000
16,Llama-3.2-3B-Instruct,A,1032,0.300388,0.777132,0.090116,3.075072,3.018901,5.290509,842.904070,583.103682,1.000000,1.000000,0.793605,0.000000,1814.0,0.049171,-0.000061,0.000000,0.000000
20,Phi-4-mini-instruct,A,1032,0.284884,0.921512,0.016473,3.306906,3.772338,6.578065,845.378876,585.419574,1.000000,1.000000,0.793605,0.000000,2558.0,0.045577,0.000837,0.000000,0.000000
8,Granite-3.3-2B-Instruct,A,1033,0.272991,0.789932,0.025169,4.860964,5.252011,10.188683,845.851888,585.890610,1.000000,1.000000,0.793804,0.000000,1058.0,0.038830,0.000514,0.000000,0.000000
12,Llama-3.2-1B-Instruct,A,1033,0.163601,0.583737,0.053243,1.788902,1.908719,4.034760,844.225557,584.343659,1.000000,1.000000,0.793804,0.000000,664.0,0.022202,0.000068,0.000000,0.000000
24,Qwen3-0.6B,A,1032,0.138700,0.699612,0.060078,1.614218,3.610133,20.282313,844.530039,584.652132,1.000000,1.000000,0.793605,0.000000,1044.0,0.000000,0.000079,0.000000,0.000000
36,SmolLM3-3B,A,1034,0.106383,0.338491,0.055126,21.702242,17.140869,23.233873,845.532882,585.581238,1.000000,1.000000,0.794004,0.000000,1590.0,0.055818,0.000831,0.000000,0.000000


Error breakdown:


,model_name,condition,verification_status,count,total,rate
0,Gemma-4-E2B-it,A,execution_error,6,1033,0.005808
1,Gemma-4-E2B-it,A,parse_error,2,1033,0.001936
2,Gemma-4-E2B-it,A,schema_error,103,1033,0.099710
3,Gemma-4-E2B-it,A,success,921,1033,0.891578
4,Gemma-4-E2B-it,A,unsafe_cartesian_product,1,1033,0.000968
...,...,...,...,...,...,...
95,Llama-3.2-3B-Instruct,C,parse_error,3,1032,0.002907
96,Llama-3.2-3B-Instruct,C,schema_error,183,1032,0.177326
97,Llama-3.2-3B-Instruct,C,success,841,1032,0.814922
98,Llama-3.2-3B-Instruct,C,unsafe_cartesian_product,3,1032,0.002907


In [ ]:
# CELL 19: Create a harder analytical subset based on SQL structure.
def sql_has_join_or_multitable(sql_text: str) -> bool:
    try:
        tree = sqlglot.parse_one(sql_text, read="sqlite")
        tables = list(tree.find_all(exp.Table))
        return len({t.name.lower() for t in tables if t.name}) >= 2
    except Exception:
        return False

def sql_has_aggregation(sql_text: str) -> bool:
    tokens = str(sql_text).lower()
    return any(x in tokens for x in ["count(", "sum(", "avg(", "min(", "max("])

def sql_has_group_order_limit(sql_text: str) -> bool:
    tokens = str(sql_text).lower()
    return any(x in tokens for x in [" group by ", " order by ", " limit "])

def sql_has_nested(sql_text: str) -> bool:
    return str(sql_text).lower().count("select") >= 2

analytic_subset = examples_for_benchmark[
    examples_for_benchmark["gold_sql"].apply(
        lambda s: (
            sql_has_join_or_multitable(s)
            or sql_has_aggregation(s)
            or sql_has_group_order_limit(s)
            or sql_has_nested(s)
        )
    )
].copy()

analytic_subset.to_csv(RESULTS_DIR / "spider_dev_analytic_subset.csv", index=False)

print(f"Analytical subset size: {len(analytic_subset)} / {len(examples_for_benchmark)}")
display(analytic_subset.head())

if "all_df" in globals() and not all_df.empty:
    analytic_ids = set(analytic_subset["example_id"].astype(int))
    analytic_outputs = all_df[all_df["example_id"].astype(int).isin(analytic_ids)].copy()
    analytic_metrics = aggregate_main_metrics(analytic_outputs)
    analytic_metrics.to_csv(RESULTS_DIR / "all_models_analytic_subset_metrics.csv", index=False)
    display(analytic_metrics.sort_values(["condition", "execution_accuracy"], ascending=[True, False]))

Analytical subset size: 877 / 1034


,example_id,db_id,question,gold_sql
0,0,concert_singer,How many singers do we have?,SELECT count(*) FROM singer
1,1,concert_singer,What is the total number of singers?,SELECT count(*) FROM singer
2,2,concert_singer,"Show name, country, age for all singers ordere...","SELECT name , country , age FROM singer ORDE..."
3,3,concert_singer,"What are the names, countries, and ages for ev...","SELECT name , country , age FROM singer ORDE..."
4,4,concert_singer,"What is the average, minimum, and maximum age ...","SELECT avg(age) , min(age) , max(age) FROM s..."


,model_name,condition,n_examples,execution_accuracy,executable_query_rate,exact_match_accuracy,median_generation_latency_sec,mean_generation_latency_sec,p95_generation_latency_sec,mean_prompt_input_chars,mean_schema_context_chars,mean_schema_compression_ratio,gold_table_recall,gold_column_recall,pruning_failure_rate,mean_model_load_gpu_reserved_delta_mb,mean_runtime_gpu_peak_delta_mb,mean_runtime_cpu_delta_mb,repair_attempt_rate,mean_repair_latency_sec
4,Gemma-4-E4B-it,A,875,0.468571,0.928000,0.116571,7.609296,8.016373,15.804267,837.025143,576.915429,1.000000,1.000000,0.830857,0.000000,8620.0,0.105756,-0.000134,0.000000,0.000000
32,Qwen3-4B-Instruct-2507,A,875,0.385143,0.961143,0.082286,4.317066,4.722476,8.703150,837.025143,576.915429,1.000000,1.000000,0.830857,0.000000,2228.0,0.064195,0.001129,0.000000,0.000000
0,Gemma-4-E2B-it,A,876,0.374429,0.885845,0.067352,7.274826,8.306043,17.363297,838.590183,578.384703,1.000000,1.000000,0.831050,0.000000,6146.0,0.132001,-0.027772,0.000000,0.000000
28,Qwen3-1.7B,A,874,0.249428,0.838673,0.092677,2.694022,3.448640,7.909010,836.831808,576.748284,1.000000,1.000000,0.830664,0.000000,2754.0,0.000000,-0.000094,0.000000,0.000000
16,Llama-3.2-3B-Instruct,A,875,0.248000,0.779429,0.098286,3.171108,3.100639,5.449595,837.025143,576.915429,1.000000,1.000000,0.830857,0.000000,1814.0,0.057993,-0.000076,0.000000,0.000000
8,Granite-3.3-2B-Instruct,A,876,0.216895,0.779680,0.025114,5.061162,5.500739,10.756718,840.507991,580.208904,1.000000,1.000000,0.831050,0.000000,1058.0,0.045789,0.000580,0.000000,0.000000
20,Phi-4-mini-instruct,A,875,0.200000,0.910857,0.012571,3.567428,3.941834,6.759249,839.944000,579.646857,1.000000,1.000000,0.830857,0.000000,2558.0,0.053754,0.000969,0.000000,0.000000
12,Llama-3.2-1B-Instruct,A,876,0.125571,0.571918,0.053653,1.815860,1.950412,4.034410,838.590183,578.384703,1.000000,1.000000,0.831050,0.000000,664.0,0.026181,0.000080,0.000000,0.000000
24,Qwen3-0.6B,A,875,0.076659,0.668571,0.065143,1.697712,3.950785,20.393358,838.942857,578.741714,1.000000,1.000000,0.830857,0.000000,1044.0,0.000000,0.000094,0.000000,0.000000
36,SmolLM3-3B,A,877,0.064994,0.316990,0.061574,21.707510,17.183379,23.260937,840.137970,579.850627,1.000000,1.000000,0.831243,0.000000,1590.0,0.050067,0.000820,0.000000,0.000000


In [ ]:
# CELL 20: Rank the models by accuracy, speed, memory, and executable-query rate.
if "all_main_metrics" not in globals() or all_main_metrics.empty:
    all_df = pd.read_parquet(RESULTS_DIR / "all_model_outputs.parquet")
    all_main_metrics = aggregate_main_metrics(all_df)

rank_cols = [
    "model_name",
    "condition",
    "n_examples",
    "execution_accuracy",
    "executable_query_rate",
    "exact_match_accuracy",
    "median_generation_latency_sec",
    "mean_generation_latency_sec",
    "p95_generation_latency_sec",
    "mean_model_load_gpu_reserved_delta_mb",
]

print("Ranking by execution accuracy, higher is better:")
acc_rank = all_main_metrics[rank_cols].sort_values(
    ["condition", "execution_accuracy"],
    ascending=[True, False],
).copy()
display(acc_rank)
acc_rank.to_csv(RESULTS_DIR / "rank_by_execution_accuracy.csv", index=False)

print("Ranking by median generation latency, lower is better:")
latency_rank = all_main_metrics[rank_cols].sort_values(
    ["condition", "median_generation_latency_sec"],
    ascending=[True, True],
).copy()
display(latency_rank)
latency_rank.to_csv(RESULTS_DIR / "rank_by_median_generation_latency.csv", index=False)

print("Ranking by executable query rate, higher is better:")
exec_rate_rank = all_main_metrics[rank_cols].sort_values(
    ["condition", "executable_query_rate"],
    ascending=[True, False],
).copy()
display(exec_rate_rank)
exec_rate_rank.to_csv(RESULTS_DIR / "rank_by_executable_query_rate.csv", index=False)

print("Ranking by model-load GPU reserved delta, lower is better:")
mem_rank = all_main_metrics[rank_cols].sort_values(
    ["condition", "mean_model_load_gpu_reserved_delta_mb"],
    ascending=[True, True],
).copy()
display(mem_rank)
mem_rank.to_csv(RESULTS_DIR / "rank_by_model_load_memory.csv", index=False)

Ranking by execution accuracy, higher is better:


,model_name,condition,n_examples,execution_accuracy,executable_query_rate,exact_match_accuracy,median_generation_latency_sec,mean_generation_latency_sec,p95_generation_latency_sec,mean_model_load_gpu_reserved_delta_mb
4,Gemma-4-E4B-it,A,1032,0.525194,0.937984,0.107558,6.705102,7.510141,14.304896,8620.0
32,Qwen3-4B-Instruct-2507,A,1032,0.450581,0.966085,0.074612,4.053564,4.496329,8.244584,2228.0
0,Gemma-4-E2B-it,A,1033,0.411423,0.891578,0.060987,7.119118,8.610090,31.575623,6146.0
28,Qwen3-1.7B,A,1029,0.315841,0.848397,0.081633,2.471303,3.246119,7.190730,2754.0
16,Llama-3.2-3B-Instruct,A,1032,0.300388,0.777132,0.090116,3.075072,3.018901,5.290509,1814.0
20,Phi-4-mini-instruct,A,1032,0.284884,0.921512,0.016473,3.306906,3.772338,6.578065,2558.0
8,Granite-3.3-2B-Instruct,A,1033,0.272991,0.789932,0.025169,4.860964,5.252011,10.188683,1058.0
12,Llama-3.2-1B-Instruct,A,1033,0.163601,0.583737,0.053243,1.788902,1.908719,4.034760,664.0
24,Qwen3-0.6B,A,1032,0.138700,0.699612,0.060078,1.614218,3.610133,20.282313,1044.0
36,SmolLM3-3B,A,1034,0.106383,0.338491,0.055126,21.702242,17.140869,23.233873,1590.0


Ranking by median generation latency, lower is better:


,model_name,condition,n_examples,execution_accuracy,executable_query_rate,exact_match_accuracy,median_generation_latency_sec,mean_generation_latency_sec,p95_generation_latency_sec,mean_model_load_gpu_reserved_delta_mb
24,Qwen3-0.6B,A,1032,0.138700,0.699612,0.060078,1.614218,3.610133,20.282313,1044.0
12,Llama-3.2-1B-Instruct,A,1033,0.163601,0.583737,0.053243,1.788902,1.908719,4.034760,664.0
28,Qwen3-1.7B,A,1029,0.315841,0.848397,0.081633,2.471303,3.246119,7.190730,2754.0
16,Llama-3.2-3B-Instruct,A,1032,0.300388,0.777132,0.090116,3.075072,3.018901,5.290509,1814.0
20,Phi-4-mini-instruct,A,1032,0.284884,0.921512,0.016473,3.306906,3.772338,6.578065,2558.0
32,Qwen3-4B-Instruct-2507,A,1032,0.450581,0.966085,0.074612,4.053564,4.496329,8.244584,2228.0
8,Granite-3.3-2B-Instruct,A,1033,0.272991,0.789932,0.025169,4.860964,5.252011,10.188683,1058.0
4,Gemma-4-E4B-it,A,1032,0.525194,0.937984,0.107558,6.705102,7.510141,14.304896,8620.0
0,Gemma-4-E2B-it,A,1033,0.411423,0.891578,0.060987,7.119118,8.610090,31.575623,6146.0
36,SmolLM3-3B,A,1034,0.106383,0.338491,0.055126,21.702242,17.140869,23.233873,1590.0


Ranking by executable query rate, higher is better:


,model_name,condition,n_examples,execution_accuracy,executable_query_rate,exact_match_accuracy,median_generation_latency_sec,mean_generation_latency_sec,p95_generation_latency_sec,mean_model_load_gpu_reserved_delta_mb
32,Qwen3-4B-Instruct-2507,A,1032,0.450581,0.966085,0.074612,4.053564,4.496329,8.244584,2228.0
4,Gemma-4-E4B-it,A,1032,0.525194,0.937984,0.107558,6.705102,7.510141,14.304896,8620.0
20,Phi-4-mini-instruct,A,1032,0.284884,0.921512,0.016473,3.306906,3.772338,6.578065,2558.0
0,Gemma-4-E2B-it,A,1033,0.411423,0.891578,0.060987,7.119118,8.610090,31.575623,6146.0
28,Qwen3-1.7B,A,1029,0.315841,0.848397,0.081633,2.471303,3.246119,7.190730,2754.0
8,Granite-3.3-2B-Instruct,A,1033,0.272991,0.789932,0.025169,4.860964,5.252011,10.188683,1058.0
16,Llama-3.2-3B-Instruct,A,1032,0.300388,0.777132,0.090116,3.075072,3.018901,5.290509,1814.0
24,Qwen3-0.6B,A,1032,0.138700,0.699612,0.060078,1.614218,3.610133,20.282313,1044.0
12,Llama-3.2-1B-Instruct,A,1033,0.163601,0.583737,0.053243,1.788902,1.908719,4.034760,664.0
36,SmolLM3-3B,A,1034,0.106383,0.338491,0.055126,21.702242,17.140869,23.233873,1590.0


Ranking by model-load GPU reserved delta, lower is better:


,model_name,condition,n_examples,execution_accuracy,executable_query_rate,exact_match_accuracy,median_generation_latency_sec,mean_generation_latency_sec,p95_generation_latency_sec,mean_model_load_gpu_reserved_delta_mb
12,Llama-3.2-1B-Instruct,A,1033,0.163601,0.583737,0.053243,1.788902,1.908719,4.034760,664.0
24,Qwen3-0.6B,A,1032,0.138700,0.699612,0.060078,1.614218,3.610133,20.282313,1044.0
8,Granite-3.3-2B-Instruct,A,1033,0.272991,0.789932,0.025169,4.860964,5.252011,10.188683,1058.0
36,SmolLM3-3B,A,1034,0.106383,0.338491,0.055126,21.702242,17.140869,23.233873,1590.0
16,Llama-3.2-3B-Instruct,A,1032,0.300388,0.777132,0.090116,3.075072,3.018901,5.290509,1814.0
32,Qwen3-4B-Instruct-2507,A,1032,0.450581,0.966085,0.074612,4.053564,4.496329,8.244584,2228.0
20,Phi-4-mini-instruct,A,1032,0.284884,0.921512,0.016473,3.306906,3.772338,6.578065,2558.0
28,Qwen3-1.7B,A,1029,0.315841,0.848397,0.081633,2.471303,3.246119,7.190730,2754.0
0,Gemma-4-E2B-it,A,1033,0.411423,0.891578,0.060987,7.119118,8.610090,31.575623,6146.0
4,Gemma-4-E4B-it,A,1032,0.525194,0.937984,0.107558,6.705102,7.510141,14.304896,8620.0


In [ ]:
# CELL 21: Compute a resource-aware ranking that balances accuracy with practical cost.
if "all_main_metrics" not in globals() or all_main_metrics.empty:
    all_df = pd.read_parquet(RESULTS_DIR / "all_model_outputs.parquet")
    all_main_metrics = aggregate_main_metrics(all_df)

score_df = all_main_metrics.copy()

# Keep the score simple and transparent:
# higher execution accuracy is good;
# higher executable-query rate is good;
# lower latency is good;
# lower load memory is good.

def minmax_good_high(s: pd.Series) -> pd.Series:
    s = s.astype(float)
    if s.max() == s.min():
        return pd.Series(np.ones(len(s)), index=s.index)
    return (s - s.min()) / (s.max() - s.min())

def minmax_good_low(s: pd.Series) -> pd.Series:
    s = s.astype(float)
    if s.max() == s.min():
        return pd.Series(np.ones(len(s)), index=s.index)
    return 1.0 - ((s - s.min()) / (s.max() - s.min()))

ranked_parts = []
for condition, part in score_df.groupby("condition"):
    part = part.copy()
    part["accuracy_score"] = minmax_good_high(part["execution_accuracy"])
    part["exec_rate_score"] = minmax_good_high(part["executable_query_rate"])
    part["latency_score"] = minmax_good_low(part["median_generation_latency_sec"])
    part["memory_score"] = minmax_good_low(part["mean_model_load_gpu_reserved_delta_mb"])

    part["resource_aware_score"] = (
        0.50 * part["accuracy_score"]
        + 0.20 * part["exec_rate_score"]
        + 0.20 * part["latency_score"]
        + 0.10 * part["memory_score"]
    )

    part = part.sort_values("resource_aware_score", ascending=False)
    part["resource_aware_rank"] = range(1, len(part) + 1)
    ranked_parts.append(part)

resource_aware_rank = pd.concat(ranked_parts, ignore_index=True)
resource_aware_rank.to_csv(RESULTS_DIR / "resource_aware_rank.csv", index=False)

display(
    resource_aware_rank[
        [
            "condition",
            "resource_aware_rank",
            "model_name",
            "resource_aware_score",
            "execution_accuracy",
            "executable_query_rate",
            "median_generation_latency_sec",
            "mean_model_load_gpu_reserved_delta_mb",
        ]
    ].sort_values(["condition", "resource_aware_rank"])
)

print("Saved final outputs to:", RESULTS_DIR)

,condition,resource_aware_rank,model_name,resource_aware_score,execution_accuracy,executable_query_rate,median_generation_latency_sec,mean_model_load_gpu_reserved_delta_mb
0,A,1,Qwen3-4B-Instruct-2507,0.866979,0.450581,0.966085,4.053564,2228.0
1,A,2,Gemma-4-E4B-it,0.840359,0.525194,0.937984,6.705102,8620.0
2,A,3,Gemma-4-E2B-it,0.716718,0.411423,0.891578,7.119118,6146.0
3,A,4,Qwen3-1.7B,0.677755,0.315841,0.848397,2.471303,2754.0
4,A,5,Phi-4-mini-instruct,0.658241,0.284884,0.921512,3.306906,2558.0
5,A,6,Llama-3.2-3B-Instruct,0.642399,0.300388,0.777132,3.075072,1814.0
6,A,7,Granite-3.3-2B-Instruct,0.605493,0.272991,0.789932,4.860964,1058.0
7,A,8,Qwen3-0.6B,0.448887,0.138700,0.699612,1.614218,1044.0
8,A,9,Llama-3.2-1B-Instruct,0.444725,0.163601,0.583737,1.788902,664.0
9,A,10,SmolLM3-3B,0.088361,0.106383,0.338491,21.702242,1590.0


Saved final outputs to: /content/drive/MyDrive/nl2sql_sqlite/results
